In [0]:
import pyspark.sql.functions as F

def main():
    # dbutils.widgets.text("odate", "2025-09-23")
    # dbutils.widgets.text("catalog", "prd")
    # dbutils.widgets.text("table_orders_bronze", "retail_orders")
    # dbutils.widgets.text("table_orders_silver", "cleaned_retail_orders")

    ODATE = dbutils.widgets.get("odate")
    CATALOG = dbutils.widgets.get("catalog")
    BRONZE_TABLE = dbutils.widgets.get("table_orders_bronze")
    SILVER_TABLE = dbutils.widgets.get("table_orders_silver")

    ORDERS_BRONZE = f"{CATALOG}.l_bronze.{BRONZE_TABLE}"
    ORDERS_SILVER = f"{CATALOG}.l_silver.{SILVER_TABLE}"
    print(f"Input Table: {ORDERS_BRONZE}")
    print(f"Output Table: {ORDERS_SILVER}")
    print(f"Order Data (Partition): {ODATE}")
    _ = (
        spark
            .read
            .format("delta")
            .table(ORDERS_BRONZE)
            .withColumn("file_name", F.element_at(F.split(F.col("file_path"), "/"), -1))
            .drop("file_path")
            .write
            .format("delta")
            .partitionBy("dat_ref_carga")
            .mode("overwrite")
            .saveAsTable(SILVER_TABLE)
    )
    print(f"Bronze Table: {SILVER_TABLE} partition {ODATE} ingested!")
        
main()